## Middleware

Middleware provides a way to more tightly control what happens inside the agent.

- Tracking agent behaviour with logging, analytics and debugging.
- Transforming prompts, tool selection and output formatting.
- Adding retires, fallbacks and early termination logic.
- Applying rate limits, guardrails and Pll detection.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPEN_AI_API_KEY"] = os.getenv("OPENAI_API_KEY")

### Summarization middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. 
Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [30]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_core.tools import tool

### MESSAGE BASED SUMMARIZATION

model = ChatGoogleGenerativeAI(
    model = "gemini-3.1-flash-lite",
    google_api_key = os.getenv("GOOGLE_API_KEY")
)

model_groq = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

agent = create_agent(
    model = model_groq,
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = model_groq,
            trigger = ("messages",10),
            keep = ("messages",4) 
        )
    ]
)

In [26]:
### Run with thread id

config = {"configurable" : {"thread_id" : "test-1"}}

In [27]:
questions = [
    "What is 2+2",
    "What is 2+5",
    "What is 2+3",
    "What is 2-4",
    "What is 14/7",
    "What is 3*9"
]

In [28]:
for q in questions : 
    response = agent.invoke({"messages" : [HumanMessage(content = q)]}, config)
    print(f"Message = {response}")
    print(f"Message length = {len(response['messages'])}")

Message = {'messages': [HumanMessage(content='What is 2+2', additional_kwargs={}, response_metadata={}, id='9c05112f-9788-4afa-8688-9b5e33caa812'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 41, 'total_tokens': 49, 'completion_time': 0.013298656, 'completion_tokens_details': None, 'prompt_time': 0.001950455, 'prompt_tokens_details': None, 'queue_time': 0.164994729, 'total_time': 0.015249111}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a005d0-bcc2-7591-a95a-e91b8a5df5c4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 8, 'total_tokens': 49})]}
Message length = 2
Message = {'messages': [HumanMessage(content='What is 2+2', additional_kwargs={}, response_metadata={}, id='9c05112f-9788-4afa-8688-9b5e33caa812'), AI

In [ ]:
## Based on token size

@tool
def search_hotels(city : str) -> str:
    """ Search hotels - returns long response to use more tokens """

    return f"""Hotels in {city} :
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym,
    2. City Inn - 4 star, $180/night, business center,
    3. Budget Stay - 3 start, $100/night, free wifi"""

model = ChatGoogleGenerativeAI(
    model = "gemini-3.1-flash-lite",
    google_api_key = os.getenv("GOOGLE_API_KEY")
)

agent = create_agent(
    model = model,
    tools = [search_hotels],
    checkpointer= InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = model,
            trigger = ("tokens",550),
            keep = ("tokens",200)
        )
    ]
)

config = {"configurable" : {"thread_id" : "test-1"}}

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4    #4 chars = 1 token

In [34]:
cities = ["paris","New york", "ahmedabad","dubai","singapore","tokyo"]

for city in cities:
    response = agent.invoke(
        {"messages" : [HumanMessage(content = f"find hotels in {city}")]},
        config = config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")


paris: ~315 tokens, 4 messages
[HumanMessage(content='find hotels in paris', additional_kwargs={}, response_metadata={}, id='96429c24-daca-4251-8c91-16131e16232f'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "paris"}'}, '__gemini_function_call_thought_signatures__': {'call_1195531': 'EnEKbwERTTIPS/eHaRR+EZnsmktP5FpvzTzLbVaeaT5oWr9vHPQhm/JocmBE3EtzOiUI2mBqJIc1IZRYZJ55s1tKB4sXbWsnHzJgkwp7jBUEdAyxu97NPilX/HrIgH9fDhB/uCh/Kww5LCxno++E+2NVgQ=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0062d-2ae6-7aa2-b917-13a95b2e4472-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'paris'}, 'id': 'call_1195531', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 16, 'total_tokens': 68, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Hotels in paris :\n   

### Fraction

In [35]:
## Based on token size

@tool
def search_hotels(city : str) -> str:
    """ Search hotels - returns long response to use more tokens """

    return f"""Hotels in {city} :
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym,
    2. City Inn - 4 star, $180/night, business center,
    3. Budget Stay - 3 start, $100/night, free wifi"""

model = ChatGoogleGenerativeAI(
    model = "gemini-3.1-flash-lite",
    google_api_key = os.getenv("GOOGLE_API_KEY")
)

agent = create_agent(
    model = model,
    tools = [search_hotels],
    checkpointer= InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = model,
            trigger = ("fraction",0.005),
            keep = ("fraction",0.002)
        )
    ]
)

config = {"configurable" : {"thread_id" : "test-1"}}

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4    #4 chars = 1 token

In [36]:
cities = ["paris","New york", "ahmedabad","dubai","singapore","tokyo"]

for city in cities:
    response = agent.invoke(
        {"messages" : [HumanMessage(content = f"find hotels in {city}")]},
        config = config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")


paris: ~163 tokens, 4 messages
[HumanMessage(content='find hotels in paris', additional_kwargs={}, response_metadata={}, id='33d42e6e-fe40-427b-a348-4efe275df9da'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_1329484': 'EnEKbwERTTIPg9Ht/Atk9qFP+O1uPfOwg9/yB9mNJOYEzJfe8erKWABpYfhzjlsJl4jyrdcdk936IQfajmFpsHG8jQDZWuimDhZ/a21YaVwZNNC5kSEByaNcGnDCqZbWSpVG4X5KayqyuqlOd/NK7an9GA=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0062f-7011-72c3-b0f4-85b0b920319d-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_1329484', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 16, 'total_tokens': 68, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Hotels in Paris :\n   

### Human in the loop middleware 

Pause agent execution for human approval

In [37]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

import os
from dotenv import load_dotenv

def read_email_tool(email_id : str) -> str:
    """Mock function to read an email by its id"""
    return f"Email content for id : {email_id}"

def send_email_tool(recipient : str,subject : str,body : str) -> str:
    """Mock fucntion to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'. "

In [38]:
model = ChatGoogleGenerativeAI(
    model = "gemini-3.1-flash-lite",
    google_api_key = os.getenv("GOOGLE_API_KEY")
)

agent = create_agent(
    model = model,
    tools = [read_email_tool,send_email_tool],
    checkpointer= InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email_tool" : {
                    "allowed_decisions" : ["approve" , "edit", "reject"]
                },

                "read_email_tool" : False
            }
        )
    ]
)

In [46]:
config = {"configurable" : {"thread_id" : "test-approve"}}

result = agent.invoke(
    {"messages" : [HumanMessage(content = "Send email to john@test.com with subject 'hello' and body 'how are you?'")]},
    config = config
)

result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='541dd69c-5f82-444b-b0b4-e6ae778bc4ca'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "how are you?", "recipient": "john@test.com", "subject": "hello"}'}, '__gemini_function_call_thought_signatures__': {'call_1471988': 'EnEKbwERTTIPdTOYuPzPDBhQDtge7pS9ZawBRfDMGNSpjwFP1b0mTd/cNjKRL9KMzkhL97kzcRMvwsnDPlbQ2YcUWfGy20ewWxBy+eE3a7FeN4L2QB0x2sFQge9aztCyu4ahzZcgs26LchG3gcGjGsrPwg=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0063c-304c-77b3-954e-55c29555a28c-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'how are you?', 'recipient': 'john@test.com', 'subject': 'hello'}, 'id': 'call_1471988', 'type': 'tool_call'}], invalid_tool_calls=[], usage_meta

In [47]:
# Approve
from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused! Approving...")

    result = agent.invoke(
        Command(
            resume = {
                "decisions" : [
                    {"type" : "approve"}
                ]
            }
        ),
        config = config
    )

    for msg in result["messages"]:
        print(type(msg).__name__)
        print(msg.content)
        print("-----")

    print(f"Result : {result['messages'][-1].text}")

Paused! Approving...
HumanMessage
Send email to john@test.com with subject 'hello' and body 'how are you?'
-----
AIMessage
[]
-----
ToolMessage
Email sent to john@test.com with subject 'hello'. 
-----
AIMessage
[{'type': 'text', 'text': "OK. I've sent the email to john@test.com.", 'extras': {'signature': 'EnEKbwERTTIP+7eFEloVfOVKVkeMBJSdmqSw1Y4gdSKbsVO7EEHAQXLuofxuAFgnNMNccxRrflnG6Kmet0hx+YLFwej2ZI+NojFOaTKpuJBZ0xQX6U/mkZGZ4VPfUMzr7L3bcaP/qss057dq4HpFxi1Ltg=='}}]
-----
HumanMessage
Send email to john@test.com with subject 'hello' and body 'how are you?'
-----
AIMessage
[]
-----
ToolMessage
Email sent to john@test.com with subject 'hello'. 
-----
AIMessage
[{'type': 'text', 'text': 'The email has been sent to john@test.com.', 'extras': {'signature': 'EnEKbwERTTIPmclf/VLZJv63mVOOGX5oDz//KrPEm3XjWVrQxrlQ+HYisSW1enH6fEnOI46QgZswwFX0Mc1A8v8YzEbyVhNm8tOoGMQQBGz2FSfxAxyBTSsFQqpb5vjWTYzgKwKaXhw2zdJtA/HW/HU9Cg=='}}]
-----
HumanMessage
Send email to john@test.com with subject 'hello' and body 'h

In [48]:
# Reject

config = {"configurable" : {"thread_id" : "test-reject"}}

result = agent.invoke(
    {"messages" : [HumanMessage(content = "Send email to john@test.com with subject 'hello' and body 'how are you?'")]},
    config = config
)

result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='69e3fc25-967e-463d-b1a1-3d1b882dd794'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "john@test.com", "subject": "hello", "body": "how are you?"}'}, '__gemini_function_call_thought_signatures__': {'call_1438667': 'EnEKbwERTTIP4ygr14oAloqi5CtRJ5Y+Ddq808DHmYKZCHOuGsWdXX1S+OSF4jGDp/gt7OzyHik2BUFcsYpVmt8Chddx2qxkh298lRDgYI1xQSTfJq8crPEphVFCG9pNgFIkfOX2Q+fQv0tt5vYmbpKK6w=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a00643-c765-7310-9659-f38c516264bc-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'hello', 'body': 'how are you?'}, 'id': 'call_1438667', 'type': 'tool_call'}], invalid_tool_calls=[], usage_meta

In [ ]:
# Approve
from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused! Rejecting...")

    result = agent.invoke(
        Command(
            resume = {
                "decisions" : [
                    {"type" : "reject"}
                ]
            }
        ),
        config = config
    )

    for msg in result["messages"]:
        print(type(msg).__name__)
        print(msg.content)
        print("-----")

    print(f"Result : {result['messages'][-1].text}")

Paused! Approving...
HumanMessage
Send email to john@test.com with subject 'hello' and body 'how are you?'
-----
AIMessage
[]
-----
ToolMessage
User rejected the tool call for `send_email_tool` with id call_1438667. The tool was not executed. Do not retry this tool call unless the user explicitly requests it.
-----
AIMessage
[{'type': 'text', 'text': 'The email could not be sent because the tool call was rejected. Please let me know if you would like me to try again or if there is anything else I can help you with.', 'extras': {'signature': 'EnEKbwERTTIPQvh0E4DZnDpIdJgHdbm2W2c3K93STYWaOmNvUAXCXuCieC3XvRQzX3dcWbLGR4YdHv5F6tB/Ty5mFV+qEtxNcwT/HsEXlfCcx9nhILQbbz0uwYM8Wz3fhSe1bFT4PMkeL4IazmuDhZt40A=='}}]
-----
Result : The email could not be sent because the tool call was rejected. Please let me know if you would like me to try again or if there is anything else I can help you with.
